# Tarea: Lógica Difusa y Algoritmos de Adaptación Social
**Materia:** Computación Bioinspirada — MIA-B  
**Autor:** Marcelo Guato (docente) | **Alumno:** [Tu nombre]  
**Fecha:** Abril 2026  

---
## Estructura
- **PARTE 1 (50%):** Lógica Difusa
  - Problema 1: Control de derrape vehicular (tracción delantera y trasera)
  - Problema 2: Aplicación en entorno laboral
- **PARTE 2 (50%):** PSO — Formación de drones

In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# Ejecutar si no están instaladas:
# !pip install scikit-fuzzy numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente ✓')

---
# PARTE 1 — Lógica Difusa (50%)

## Problema 1: Control de Derrape Vehicular

### Análisis del problema
Cuando un vehículo derrapa, el conductor debe actuar de forma coordinada:
- **Soltar el acelerador** gradualmente
- **Frenar suavemente** (sin bloquear ruedas)
- **Girar el volante** en dirección opuesta al derrape

Las variables antecedentes y consecuentes difieren entre tracción delantera y trasera:

| | Tracción Delantera (Caso A) | Tracción Trasera (Caso B) |
|---|---|---|
| Antecedentes | Ángulo derrape, velocidad, humedad | Ángulo derrape, aceleración, humedad |
| Consecuentes | Corrección volante, frenado | Corrección volante, reducción acelerador |

### Universo del Discurso
- **Ángulo de derrape:** 0° a 45° (0 = recto, 45 = derrape severo)
- **Velocidad del vehículo:** 0 a 120 km/h
- **Humedad de calzada:** 0 a 100% (0 = seca, 100 = muy mojada)
- **Corrección de volante:** 0 a 45° (giro de corrección)
- **Intensidad de frenado:** 0 a 100% (0 = sin freno, 100 = freno máximo)

---
### CASO A — Vehículo de Tracción Delantera

En tracción delantera, las ruedas delanteras traccionan Y dirigen. En un derrape:
- Se pierde adherencia principalmente en el eje trasero
- La corrección principal es **girar el volante hacia donde va el derrape**
- El frenado debe ser **muy suave** para no bloquear las ruedas delanteras

In [ ]:
# ============================================================
# CASO A: VEHÍCULO DE TRACCIÓN DELANTERA
# ============================================================

# --- 1. DEFINICIÓN DE UNIVERSOS DE DISCURSO ---
angulo_derrape_range  = np.arange(0, 46, 1)   # 0° a 45°
velocidad_range       = np.arange(0, 121, 1)  # 0 a 120 km/h
humedad_range         = np.arange(0, 101, 1)  # 0% a 100%
correccion_vol_range  = np.arange(0, 46, 1)   # 0° a 45°
frenado_range         = np.arange(0, 101, 1)  # 0% a 100%

# --- 2. VARIABLES LINGÜÍSTICAS (Antecedentes y Consecuentes) ---
angulo   = ctrl.Antecedent(angulo_derrape_range,  'angulo_derrape')
velocidad = ctrl.Antecedent(velocidad_range,       'velocidad')
humedad  = ctrl.Antecedent(humedad_range,          'humedad_calzada')

correccion = ctrl.Consequent(correccion_vol_range, 'correccion_volante')
frenado    = ctrl.Consequent(frenado_range,        'intensidad_frenado')

# --- 3. FUNCIONES DE PERTENENCIA ---

# Ángulo de derrape: leve / moderado / severo
angulo['leve']     = fuzz.trimf(angulo.universe, [0, 0, 15])
angulo['moderado'] = fuzz.trimf(angulo.universe, [10, 22, 35])
angulo['severo']   = fuzz.trapmf(angulo.universe, [28, 38, 45, 45])

# Velocidad: baja / media / alta
velocidad['baja']  = fuzz.trapmf(velocidad.universe, [0, 0, 30, 50])
velocidad['media'] = fuzz.trimf(velocidad.universe, [40, 70, 100])
velocidad['alta']  = fuzz.trapmf(velocidad.universe, [90, 105, 120, 120])

# Humedad calzada: seca / húmeda / muy_húmeda
humedad['seca']       = fuzz.trapmf(humedad.universe, [0, 0, 20, 35])
humedad['humeda']     = fuzz.trimf(humedad.universe, [25, 50, 75])
humedad['muy_humeda'] = fuzz.trapmf(humedad.universe, [65, 80, 100, 100])

# Corrección de volante (consecuente): suave / moderada / brusca
correccion['suave']    = fuzz.trapmf(correccion.universe, [0, 0, 8, 15])
correccion['moderada'] = fuzz.trimf(correccion.universe, [10, 22, 35])
correccion['brusca']   = fuzz.trapmf(correccion.universe, [28, 38, 45, 45])

# Frenado (consecuente): sin_freno / suave / moderado
# En tracción delantera el frenado NUNCA debe ser brusco
frenado['sin_freno'] = fuzz.trapmf(frenado.universe, [0, 0, 5, 15])
frenado['suave']     = fuzz.trimf(frenado.universe, [10, 30, 50])
frenado['moderado']  = fuzz.trapmf(frenado.universe, [40, 60, 100, 100])

# --- Visualizar funciones de pertenencia ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Caso A — Tracción Delantera: Funciones de Pertenencia', fontsize=14, fontweight='bold')

angulo.view(ax=axes[0,0])
velocidad.view(ax=axes[0,1])
humedad.view(ax=axes[0,2])
correccion.view(ax=axes[1,0])
frenado.view(ax=axes[1,1])
axes[1,2].axis('off')

plt.tight_layout()
plt.show()
print('Funciones de pertenencia Caso A definidas ✓')

In [ ]:
# --- 4. REGLAS DIFUSAS — CASO A (Tracción Delantera) ---
# Lógica: En TD el derrape ocurre por pérdida de tracción trasera.
# La corrección del volante es la acción principal; el freno debe ser muy suave.

reglas_A = [
    # Derrape leve → corrección mínima, sin freno agresivo
    ctrl.Rule(angulo['leve'] & velocidad['baja'],              (correccion['suave'],    frenado['sin_freno'])),
    ctrl.Rule(angulo['leve'] & velocidad['media'],             (correccion['suave'],    frenado['suave'])),
    ctrl.Rule(angulo['leve'] & velocidad['alta'],              (correccion['moderada'], frenado['suave'])),

    # Derrape moderado → corrección proporcional a velocidad y humedad
    ctrl.Rule(angulo['moderado'] & humedad['seca'],            (correccion['moderada'], frenado['suave'])),
    ctrl.Rule(angulo['moderado'] & humedad['humeda'],          (correccion['moderada'], frenado['suave'])),
    ctrl.Rule(angulo['moderado'] & humedad['muy_humeda'],      (correccion['brusca'],   frenado['sin_freno'])),
    ctrl.Rule(angulo['moderado'] & velocidad['alta'],          (correccion['brusca'],   frenado['suave'])),

    # Derrape severo → máxima corrección, frenar muy suave o nada
    ctrl.Rule(angulo['severo'] & humedad['seca'],              (correccion['brusca'],   frenado['suave'])),
    ctrl.Rule(angulo['severo'] & humedad['humeda'],            (correccion['brusca'],   frenado['suave'])),
    ctrl.Rule(angulo['severo'] & humedad['muy_humeda'],        (correccion['brusca'],   frenado['sin_freno'])),
    ctrl.Rule(angulo['severo'] & velocidad['alta'],            (correccion['brusca'],   frenado['sin_freno'])),

    # Velocidad alta siempre requiere mayor corrección
    ctrl.Rule(velocidad['alta'] & humedad['muy_humeda'],       (correccion['brusca'],   frenado['sin_freno'])),
]

# --- 5. CONSTRUCCIÓN DEL SISTEMA DE CONTROL DIFUSO ---
sistema_A = ctrl.ControlSystem(reglas_A)
sim_A     = ctrl.ControlSystemSimulation(sistema_A)

print('Sistema de control difuso Caso A construido ✓')
print(f'Total de reglas: {len(reglas_A)}')

In [ ]:
# --- 6. SIMULACIÓN Y DEFUZZIFICACIÓN — CASO A ---
# Escenario: lluvia moderada, velocidad media, derrape moderado

escenarios_A = [
    {'angulo': 8,  'velocidad': 40, 'humedad': 30,  'desc': 'Derrape leve, seco, baja velocidad'},
    {'angulo': 20, 'velocidad': 70, 'humedad': 60,  'desc': 'Derrape moderado, húmedo, media velocidad'},
    {'angulo': 35, 'velocidad': 90, 'humedad': 85,  'desc': 'Derrape severo, muy húmedo, alta velocidad'},
    {'angulo': 25, 'velocidad': 50, 'humedad': 90,  'desc': 'Derrape moderado, muy húmedo (lluvia)'},
]

print('='*65)
print('CASO A — TRACCIÓN DELANTERA: Resultados de Simulación')
print('='*65)

for esc in escenarios_A:
    sim_A.input['angulo_derrape']   = esc['angulo']
    sim_A.input['velocidad']        = esc['velocidad']
    sim_A.input['humedad_calzada']  = esc['humedad']
    sim_A.compute()

    corr_val   = sim_A.output['correccion_volante']
    fren_val   = sim_A.output['intensidad_frenado']

    print(f'\n📍 Escenario: {esc["desc"]}')
    print(f'   Entradas → Ángulo: {esc["angulo"]}°  |  Velocidad: {esc["velocidad"]} km/h  |  Humedad: {esc["humedad"]}%')
    print(f'   Salidas  → Corrección volante: {corr_val:.1f}°  |  Frenado: {fren_val:.1f}%')
    if fren_val < 20:
        consejo_freno = '⚠️  Suelta el freno o frena muy suave'
    elif fren_val < 50:
        consejo_freno = '🟡 Frenado suave, controlado'
    else:
        consejo_freno = '🔴 Frenado moderado'
    print(f'   Consejo  → {consejo_freno}')

print('\n' + '='*65)

In [ ]:
# Visualización del escenario 2 (más representativo)
sim_A.input['angulo_derrape']  = 20
sim_A.input['velocidad']       = 70
sim_A.input['humedad_calzada'] = 60
sim_A.compute()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Caso A — Defuzzificación: Derrape moderado, húmedo, 70 km/h', fontsize=12)
correccion.view(sim=sim_A, ax=axes[0])
frenado.view(sim=sim_A, ax=axes[1])
plt.tight_layout()
plt.show()

---
### CASO B — Vehículo de Tracción Trasera

En tracción trasera, el motor empuja desde las ruedas traseras. En un derrape:
- El derrape trasero es más fácil de provocar y más difícil de controlar
- La acción principal es **reducir el acelerador** (soltar el gas)
- La corrección del volante debe hacerse **en la misma dirección del derrape** (contravolante)
- El frenado se evita casi completamente durante el derrape activo

In [ ]:
# ============================================================
# CASO B: VEHÍCULO DE TRACCIÓN TRASERA
# ============================================================

# Universos de discurso
angulo_B     = ctrl.Antecedent(np.arange(0, 46, 1),  'angulo_derrape')
aceleracion  = ctrl.Antecedent(np.arange(0, 101, 1), 'aceleracion')    # % pedal acelerador
humedad_B    = ctrl.Antecedent(np.arange(0, 101, 1), 'humedad_calzada')

correccion_B  = ctrl.Consequent(np.arange(0, 46, 1),  'correccion_volante')
reduccion_acc = ctrl.Consequent(np.arange(0, 101, 1), 'reduccion_acelerador')  # % reducción

# Funciones de pertenencia
angulo_B['leve']     = fuzz.trimf(angulo_B.universe, [0, 0, 15])
angulo_B['moderado'] = fuzz.trimf(angulo_B.universe, [10, 22, 35])
angulo_B['severo']   = fuzz.trapmf(angulo_B.universe, [28, 38, 45, 45])

aceleracion['baja']   = fuzz.trapmf(aceleracion.universe, [0, 0, 20, 40])
aceleracion['media']  = fuzz.trimf(aceleracion.universe, [30, 55, 75])
aceleracion['alta']   = fuzz.trapmf(aceleracion.universe, [65, 80, 100, 100])

humedad_B['seca']       = fuzz.trapmf(humedad_B.universe, [0, 0, 20, 35])
humedad_B['humeda']     = fuzz.trimf(humedad_B.universe, [25, 50, 75])
humedad_B['muy_humeda'] = fuzz.trapmf(humedad_B.universe, [65, 80, 100, 100])

correccion_B['suave']    = fuzz.trapmf(correccion_B.universe, [0, 0, 8, 15])
correccion_B['moderada'] = fuzz.trimf(correccion_B.universe, [10, 22, 35])
correccion_B['brusca']   = fuzz.trapmf(correccion_B.universe, [28, 38, 45, 45])

# Reducción del acelerador: poca / media / total
reduccion_acc['poca']  = fuzz.trapmf(reduccion_acc.universe, [0, 0, 15, 30])
reduccion_acc['media'] = fuzz.trimf(reduccion_acc.universe, [25, 50, 75])
reduccion_acc['total'] = fuzz.trapmf(reduccion_acc.universe, [65, 80, 100, 100])

# Reglas — Caso B (Tracción Trasera)
# CLAVE: En TT, la reducción del acelerador es la acción primaria
reglas_B = [
    ctrl.Rule(angulo_B['leve'] & aceleracion['baja'],              (correccion_B['suave'],    reduccion_acc['poca'])),
    ctrl.Rule(angulo_B['leve'] & aceleracion['media'],             (correccion_B['suave'],    reduccion_acc['media'])),
    ctrl.Rule(angulo_B['leve'] & aceleracion['alta'],              (correccion_B['moderada'], reduccion_acc['media'])),

    ctrl.Rule(angulo_B['moderado'] & aceleracion['baja'],          (correccion_B['moderada'], reduccion_acc['media'])),
    ctrl.Rule(angulo_B['moderado'] & aceleracion['alta'],          (correccion_B['brusca'],   reduccion_acc['total'])),
    ctrl.Rule(angulo_B['moderado'] & humedad_B['muy_humeda'],      (correccion_B['brusca'],   reduccion_acc['total'])),
    ctrl.Rule(angulo_B['moderado'] & humedad_B['humeda'],          (correccion_B['moderada'], reduccion_acc['media'])),

    ctrl.Rule(angulo_B['severo'] & aceleracion['alta'],            (correccion_B['brusca'],   reduccion_acc['total'])),
    ctrl.Rule(angulo_B['severo'] & humedad_B['seca'],              (correccion_B['brusca'],   reduccion_acc['media'])),
    ctrl.Rule(angulo_B['severo'] & humedad_B['muy_humeda'],        (correccion_B['brusca'],   reduccion_acc['total'])),
    ctrl.Rule(angulo_B['severo'] & humedad_B['humeda'],            (correccion_B['brusca'],   reduccion_acc['total'])),

    ctrl.Rule(aceleracion['alta'] & humedad_B['muy_humeda'],       (correccion_B['brusca'],   reduccion_acc['total'])),
]

sistema_B = ctrl.ControlSystem(reglas_B)
sim_B     = ctrl.ControlSystemSimulation(sistema_B)

print('Sistema de control difuso Caso B construido ✓')

# Simulación
escenarios_B = [
    {'angulo': 8,  'aceleracion': 30, 'humedad': 20,  'desc': 'Derrape leve, seco, aceleración baja'},
    {'angulo': 22, 'aceleracion': 70, 'humedad': 55,  'desc': 'Derrape moderado, húmedo, aceleración alta'},
    {'angulo': 38, 'aceleracion': 85, 'humedad': 90,  'desc': 'Derrape severo, muy húmedo, aceleración alta'},
    {'angulo': 18, 'aceleracion': 60, 'humedad': 80,  'desc': 'Derrape moderado en lluvia'},
]

print('='*65)
print('CASO B — TRACCIÓN TRASERA: Resultados de Simulación')
print('='*65)

for esc in escenarios_B:
    sim_B.input['angulo_derrape']       = esc['angulo']
    sim_B.input['aceleracion']          = esc['aceleracion']
    sim_B.input['humedad_calzada']      = esc['humedad']
    sim_B.compute()

    corr_val = sim_B.output['correccion_volante']
    red_val  = sim_B.output['reduccion_acelerador']

    print(f'\n📍 {esc["desc"]}')
    print(f'   Entradas → Ángulo: {esc["angulo"]}°  |  Aceleración: {esc["aceleracion"]}%  |  Humedad: {esc["humedad"]}%')
    print(f'   Salidas  → Corrección volante: {corr_val:.1f}°  |  Reducción acelerador: {red_val:.1f}%')
    if red_val > 70:
        consejo = '🔴 Suelta el acelerador completamente'
    elif red_val > 40:
        consejo = '🟡 Reduce el acelerador gradualmente'
    else:
        consejo = '🟢 Ajuste mínimo del acelerador'
    print(f'   Consejo  → {consejo}')

print('\n' + '='*65)

In [ ]:
# Visualización comparativa A vs B
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Comparativa: Tracción Delantera vs Trasera — Derrape moderado en lluvia', fontsize=13, fontweight='bold')

# Caso A
sim_A.input['angulo_derrape']  = 22
sim_A.input['velocidad']       = 65
sim_A.input['humedad_calzada'] = 75
sim_A.compute()
correccion.view(sim=sim_A, ax=axes[0,0])
axes[0,0].set_title(f'Caso A — Corrección volante: {sim_A.output["correccion_volante"]:.1f}°')
frenado.view(sim=sim_A, ax=axes[0,1])
axes[0,1].set_title(f'Caso A — Frenado: {sim_A.output["intensidad_frenado"]:.1f}%')

# Caso B
sim_B.input['angulo_derrape']   = 22
sim_B.input['aceleracion']      = 60
sim_B.input['humedad_calzada']  = 75
sim_B.compute()
correccion_B.view(sim=sim_B, ax=axes[1,0])
axes[1,0].set_title(f'Caso B — Corrección volante: {sim_B.output["correccion_volante"]:.1f}°')
reduccion_acc.view(sim=sim_B, ax=axes[1,1])
axes[1,1].set_title(f'Caso B — Reducción acelerador: {sim_B.output["reduccion_acelerador"]:.1f}%')

plt.tight_layout()
plt.show()

---
## Problema 2: Aplicación en Entorno Laboral

### Sistema Difuso de Evaluación de Riesgo Crediticio (Área Financiera/Bancaria)

**Contexto:** En entidades bancarias o financieras, la evaluación del riesgo de un crédito no es binaria (aprobado/rechazado). Existen múltiples factores que en conjunto determinan el nivel de riesgo, y muchos de ellos son difusos por naturaleza (¿qué es un ingreso "alto"? ¿cuándo una deuda es "excesiva"?).

**Universo del discurso:**
- **Ingreso mensual:** 0 a 10.000 USD
- **Nivel de endeudamiento:** 0 a 100% (deuda/ingreso)
- **Historial de crédito:** 0 a 100 puntos (score)
- **Nivel de riesgo:** 0 a 100 (salida, 0=sin riesgo, 100=riesgo máximo)

In [ ]:
# ============================================================
# PROBLEMA 2: EVALUACIÓN DE RIESGO CREDITICIO
# ============================================================

# Universos
ingreso       = ctrl.Antecedent(np.arange(0, 10001, 100), 'ingreso_mensual')
endeudamiento = ctrl.Antecedent(np.arange(0, 101, 1),     'nivel_endeudamiento')
historial     = ctrl.Antecedent(np.arange(0, 101, 1),     'historial_credito')
riesgo        = ctrl.Consequent(np.arange(0, 101, 1),     'nivel_riesgo')

# Funciones de pertenencia
ingreso['bajo']  = fuzz.trapmf(ingreso.universe, [0, 0, 1500, 3000])
ingreso['medio'] = fuzz.trimf(ingreso.universe,  [2000, 4500, 7000])
ingreso['alto']  = fuzz.trapmf(ingreso.universe, [6000, 8000, 10000, 10000])

endeudamiento['bajo']  = fuzz.trapmf(endeudamiento.universe, [0, 0, 20, 35])
endeudamiento['medio'] = fuzz.trimf(endeudamiento.universe,  [25, 50, 70])
endeudamiento['alto']  = fuzz.trapmf(endeudamiento.universe, [60, 75, 100, 100])

historial['malo']   = fuzz.trapmf(historial.universe, [0, 0, 30, 50])
historial['regular']= fuzz.trimf(historial.universe,  [40, 60, 80])
historial['bueno']  = fuzz.trapmf(historial.universe, [70, 85, 100, 100])

riesgo['bajo']   = fuzz.trapmf(riesgo.universe, [0, 0, 20, 35])
riesgo['medio']  = fuzz.trimf(riesgo.universe,  [25, 50, 75])
riesgo['alto']   = fuzz.trapmf(riesgo.universe, [65, 80, 100, 100])

# Reglas difusas
reglas_credito = [
    ctrl.Rule(ingreso['alto']  & endeudamiento['bajo']  & historial['bueno'],   riesgo['bajo']),
    ctrl.Rule(ingreso['medio'] & endeudamiento['bajo']  & historial['bueno'],   riesgo['bajo']),
    ctrl.Rule(ingreso['alto']  & endeudamiento['medio'] & historial['bueno'],   riesgo['bajo']),
    ctrl.Rule(ingreso['medio'] & endeudamiento['medio'] & historial['regular'], riesgo['medio']),
    ctrl.Rule(ingreso['alto']  & endeudamiento['alto']  & historial['regular'], riesgo['medio']),
    ctrl.Rule(ingreso['bajo']  & endeudamiento['bajo']  & historial['bueno'],   riesgo['medio']),
    ctrl.Rule(ingreso['bajo']  & endeudamiento['medio'],                         riesgo['alto']),
    ctrl.Rule(ingreso['bajo']  & endeudamiento['alto'],                          riesgo['alto']),
    ctrl.Rule(historial['malo'],                                                  riesgo['alto']),
    ctrl.Rule(ingreso['medio'] & endeudamiento['alto']  & historial['malo'],    riesgo['alto']),
    ctrl.Rule(ingreso['bajo']  & endeudamiento['alto']  & historial['malo'],    riesgo['alto']),
]

sistema_credito = ctrl.ControlSystem(reglas_credito)
sim_credito     = ctrl.ControlSystemSimulation(sistema_credito)

# Simulación de perfiles de clientes
clientes = [
    {'ingreso': 6500, 'deuda': 20, 'historial': 88, 'nombre': 'Cliente A — Perfil excelente'},
    {'ingreso': 3200, 'deuda': 55, 'historial': 62, 'nombre': 'Cliente B — Perfil medio'},
    {'ingreso': 1200, 'deuda': 80, 'historial': 25, 'nombre': 'Cliente C — Perfil de riesgo'},
    {'ingreso': 4500, 'deuda': 35, 'historial': 72, 'nombre': 'Cliente D — Perfil aceptable'},
]

print('='*65)
print('PROBLEMA 2 — RIESGO CREDITICIO: Resultados')
print('='*65)

for c in clientes:
    sim_credito.input['ingreso_mensual']      = c['ingreso']
    sim_credito.input['nivel_endeudamiento']  = c['deuda']
    sim_credito.input['historial_credito']    = c['historial']
    sim_credito.compute()
    r = sim_credito.output['nivel_riesgo']
    decision = '✅ APROBADO' if r < 35 else ('⚠️  CONDICIONAL' if r < 65 else '❌ RECHAZADO')
    print(f'\n👤 {c["nombre"]}')
    print(f'   Ingreso: ${c["ingreso"]}  |  Deuda: {c["deuda"]}%  |  Score: {c["historial"]}')
    print(f'   Riesgo calculado: {r:.1f}/100  →  {decision}')

print('\n' + '='*65)

---
# PARTE 2 — Algoritmo PSO: Formación de Drones (50%)

## Descripción del Problema
Se implementa el algoritmo PSO para optimizar las posiciones de una flota de drones manteniendo formaciones geométricas dadas dentro de un área de vuelo definida.

**Cada partícula** representa una configuración completa de posiciones de todos los drones.  
**La función de fitness** mide cuánto se aleja la configuración de la formación objetivo, penalizando colisiones y salidas del área.

In [ ]:
# ============================================================
# PARTE 2 — PSO: FORMACIÓN DE DRONES
# Parámetros del problema y del algoritmo
# ============================================================

# --- Parámetros del problema ---
NUM_DRONES           = 6      # Número de drones en la flota
DIMENSIONS           = 2      # Espacio 2D (x, y)
AREA_WIDTH           = 100    # Ancho del área de vuelo (metros)
AREA_HEIGHT          = 100    # Alto del área de vuelo (metros)
MIN_SAFE_DISTANCE    = 5.0    # Distancia mínima de seguridad entre drones (metros)
FORMATION_DISTANCE_L = 15.0   # Distancia entre drones en la formación (metros)

# --- Parámetros del algoritmo PSO ---
NUM_PARTICLES  = 80           # Número de partículas en el enjambre
MAX_ITERATIONS = 200          # Iteraciones máximas

# Coeficientes PSO
W   = 0.5    # Inercia: controla cuánto sigue la partícula su trayectoria previa
C1  = 1.5    # Coeficiente cognitivo: atracción hacia el mejor personal
C2  = 1.5    # Coeficiente social: atracción hacia el mejor global
V_MAX = 10.0  # Velocidad máxima permitida

print('Parámetros configurados:')
print(f'  Drones: {NUM_DRONES}  |  Área: {AREA_WIDTH}x{AREA_HEIGHT}m  |  Dist. mínima: {MIN_SAFE_DISTANCE}m')
print(f'  Partículas: {NUM_PARTICLES}  |  Iteraciones: {MAX_ITERATIONS}')
print(f'  PSO → W={W}, C1={C1}, C2={C2}, V_max={V_MAX}')

In [ ]:
# ============================================================
# DEFINICIÓN DE FORMACIONES OBJETIVO
# Retorna posiciones relativas de los drones respecto al centroide
# ============================================================

def get_target_formation_relative_positions(formation_type="line"):
    """
    Calcula las posiciones relativas ideales de los drones
    respecto al centroide de la formación.
    
    Formaciones soportadas:
    - 'line'     : Línea horizontal
    - 'triangle' : Triángulo equilátero (requiere NUM_DRONES==3)
    - 'H'        : Forma de H (requiere NUM_DRONES==6)
    - 'rectangle': Rectángulo (requiere NUM_DRONES par >= 4)
    """
    targets = np.zeros((NUM_DRONES, DIMENSIONS))
    L = FORMATION_DISTANCE_L  # alias corto

    if formation_type == "line":
        # Todos los drones en fila horizontal centrada en el origen
        start_x = -(NUM_DRONES - 1) * L / 2.0
        for i in range(NUM_DRONES):
            targets[i, 0] = start_x + i * L
            targets[i, 1] = 0

    elif formation_type == "triangle" and NUM_DRONES == 3:
        # Triángulo equilátero
        h = (np.sqrt(3) / 2) * L
        targets[0, :] = [0,        h * 2/3]   # Vértice superior
        targets[1, :] = [-L / 2,  -h / 3]    # Base izquierda
        targets[2, :] = [ L / 2,  -h / 3]    # Base derecha

    elif formation_type == "H" and NUM_DRONES == 6:
        # Formación en H: 2 columnas verticales + barra horizontal central
        # Columna izquierda: drones 0, 1, 2  |  Columna derecha: drones 3, 4, 5
        # La barra de la H la forman drones 1 y 4 (los del medio)
        col_sep = L * 1.5   # Separación horizontal entre columnas
        row_sep = L          # Separación vertical entre filas

        # Columna izquierda
        targets[0, :] = [-col_sep / 2,  row_sep]   # Superior izq
        targets[1, :] = [-col_sep / 2,  0       ]  # Medio izq (barra H)
        targets[2, :] = [-col_sep / 2, -row_sep ]  # Inferior izq

        # Columna derecha
        targets[3, :] = [ col_sep / 2,  row_sep]   # Superior der
        targets[4, :] = [ col_sep / 2,  0      ]   # Medio der (barra H)
        targets[5, :] = [ col_sep / 2, -row_sep]   # Inferior der

    elif formation_type == "rectangle" and NUM_DRONES >= 4 and NUM_DRONES % 2 == 0:
        # Rectángulo: drones distribuidos en perímetro de un rectángulo
        half_n  = NUM_DRONES // 2
        width   = (half_n - 1) * L
        height  = L * 1.2

        for i in range(half_n):
            x = -width / 2 + i * L
            targets[i,           :] = [x,  height / 2]   # Fila superior
            targets[i + half_n,  :] = [x, -height / 2]   # Fila inferior

    else:
        raise ValueError(f"Formación '{formation_type}' no soportada para {NUM_DRONES} drones.")

    return targets

# Verificar formaciones
for f in ['line', 'H', 'rectangle']:
    pos = get_target_formation_relative_positions(f)
    print(f'Formación "{f}" generada correctamente. Shape: {pos.shape}')

In [ ]:
# ============================================================
# FUNCIÓN DE FITNESS
# Evalúa qué tan buena es una configuración de drones
# Minimiza: error_formación + penalización_colisión + penalización_límites
# ============================================================

def fitness_function(particle_position, target_relative_pos):
    """
    Calcula el fitness (costo) de una partícula.
    
    Args:
        particle_position   : vector 1D de tamaño NUM_DRONES * DIMENSIONS
        target_relative_pos : posiciones relativas ideales (NUM_DRONES x DIMENSIONS)
    
    Returns:
        total_fitness (float): valor a minimizar (0 = solución perfecta)
    """
    # Reshape a matriz de posiciones (NUM_DRONES, 2)
    drone_positions = particle_position.reshape((NUM_DRONES, DIMENSIONS))

    # 1. ERROR DE FORMACIÓN (e_form)
    # Compara la posición actual de cada dron con su posición ideal
    # respecto al centroide actual del grupo
    current_centroid = np.mean(drone_positions, axis=0)
    e_form = 0.0
    for i in range(NUM_DRONES):
        ideal_pos_i = current_centroid + target_relative_pos[i, :]
        e_form += np.sum((drone_positions[i, :] - ideal_pos_i) ** 2)

    # 2. PENALIZACIÓN POR COLISIÓN (p_coll)
    # Si dos drones están más cerca que MIN_SAFE_DISTANCE, se penaliza
    p_coll = 0.0
    collision_penalty_value = 10000  # Penalización alta y fija por colisión
    for i in range(NUM_DRONES):
        for j in range(i + 1, NUM_DRONES):
            dist = np.sqrt(np.sum((drone_positions[i, :] - drone_positions[j, :]) ** 2))
            if dist < MIN_SAFE_DISTANCE:
                p_coll += collision_penalty_value

    # 3. PENALIZACIÓN POR LÍMITES DEL ÁREA (p_boundary)
    # Si un dron sale del área de vuelo definida, se penaliza
    p_boundary = 0.0
    boundary_penalty_value = 1000
    for i in range(NUM_DRONES):
        if not (0 <= drone_positions[i, 0] <= AREA_WIDTH and
                0 <= drone_positions[i, 1] <= AREA_HEIGHT):
            p_boundary += boundary_penalty_value

    # PESOS para cada componente del fitness
    w1 = 1.0   # Peso del error de formación (principal objetivo)
    w2 = 1.0   # Peso de penalización por colisión
    w3 = 0.5   # Peso de penalización por límites (menor prioridad)

    total_fitness = w1 * e_form + w2 * p_coll + w3 * p_boundary
    return total_fitness

print('Función de fitness definida ✓')

In [ ]:
# ============================================================
# IMPLEMENTACIÓN DEL ALGORITMO PSO
# ============================================================

def run_pso(formation_type):
    """
    Ejecuta el algoritmo PSO para una formación dada.
    
    Flujo PSO:
    1. Inicializar posiciones y velocidades aleatorias
    2. Evaluar fitness inicial
    3. En cada iteración:
       a. Para cada partícula: calcular fitness
       b. Actualizar pBest (mejor personal) y gBest (mejor global)
       c. Actualizar velocidad: v = W*v + C1*r1*(pBest-x) + C2*r2*(gBest-x)
       d. Actualizar posición: x = x + v
       e. Aplicar límites del área (clipping)
    4. Retornar la mejor solución encontrada
    """
    print(f'\n🚁 Ejecutando PSO — Formación: {formation_type.upper()}')
    print('-' * 50)

    # Obtener posiciones relativas objetivo para la formación
    target_rel_pos = get_target_formation_relative_positions(formation_type)

    # --- INICIALIZACIÓN ---
    # Posiciones aleatorias dentro del área de vuelo
    # Cada fila = una partícula = vector con posiciones de todos los drones
    particles_pos = np.random.rand(NUM_PARTICLES, NUM_DRONES * DIMENSIONS)
    particles_pos[:, ::2]  *= AREA_WIDTH   # Coordenadas X
    particles_pos[:, 1::2] *= AREA_HEIGHT  # Coordenadas Y

    # Velocidades iniciales pequeñas
    particles_vel = np.random.rand(NUM_PARTICLES, NUM_DRONES * DIMENSIONS) * V_MAX * 0.1

    # Mejor posición personal de cada partícula
    pbest_pos     = np.copy(particles_pos)
    pbest_fitness = np.full(NUM_PARTICLES, float('inf'))

    # Mejor posición global del enjambre
    gbest_pos     = np.zeros(NUM_DRONES * DIMENSIONS)
    gbest_fitness = float('inf')

    fitness_history = []  # Historial para graficar convergencia

    # --- BUCLE PRINCIPAL PSO ---
    for iter_num in range(MAX_ITERATIONS):

        # Evaluar fitness de cada partícula y actualizar pBest y gBest
        for i in range(NUM_PARTICLES):
            current_fitness = fitness_function(particles_pos[i], target_rel_pos)

            # Actualizar mejor personal (pBest)
            if current_fitness < pbest_fitness[i]:
                pbest_fitness[i] = current_fitness
                pbest_pos[i]     = np.copy(particles_pos[i])

            # Actualizar mejor global (gBest)
            if current_fitness < gbest_fitness:
                gbest_fitness = current_fitness
                gbest_pos     = np.copy(particles_pos[i])

        fitness_history.append(gbest_fitness)

        # Actualizar velocidades y posiciones
        for i in range(NUM_PARTICLES):
            r1 = np.random.rand(NUM_DRONES * DIMENSIONS)  # Aleatoriedad cognitiva
            r2 = np.random.rand(NUM_DRONES * DIMENSIONS)  # Aleatoriedad social

            # Componente cognitiva: atracción al mejor personal
            cognitive_vel = C1 * r1 * (pbest_pos[i] - particles_pos[i])
            # Componente social: atracción al mejor global
            social_vel    = C2 * r2 * (gbest_pos    - particles_pos[i])

            # Actualización de velocidad (fórmula PSO)
            particles_vel[i] = W * particles_vel[i] + cognitive_vel + social_vel

            # Limitar velocidad máxima
            particles_vel[i] = np.clip(particles_vel[i], -V_MAX, V_MAX)

            # Actualizar posición
            particles_pos[i] += particles_vel[i]

            # Aplicar límites del área de vuelo (clipping)
            particles_pos[i, ::2]  = np.clip(particles_pos[i, ::2],  0, AREA_WIDTH)
            particles_pos[i, 1::2] = np.clip(particles_pos[i, 1::2], 0, AREA_HEIGHT)

        if (iter_num + 1) % 50 == 0:
            print(f'  Iteración {iter_num+1:3d}/{MAX_ITERATIONS} → Mejor Fitness: {gbest_fitness:.2f}')

    print(f'\n✅ Fitness final: {gbest_fitness:.4f}')

    final_positions = gbest_pos.reshape((NUM_DRONES, DIMENSIONS))

    # Verificar colisiones en solución final
    colisiones = 0
    for i in range(NUM_DRONES):
        for j in range(i + 1, NUM_DRONES):
            d = np.linalg.norm(final_positions[i] - final_positions[j])
            if d < MIN_SAFE_DISTANCE:
                colisiones += 1
    print(f'   Colisiones en solución final: {colisiones}')

    return final_positions, fitness_history, target_rel_pos

print('Función PSO definida ✓')

In [ ]:
# ============================================================
# FUNCIÓN DE VISUALIZACIÓN
# ============================================================

def visualize_results(final_positions, fitness_history, target_rel_pos, formation_type):
    """Genera los gráficos de convergencia y formación final."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'PSO — Formación {formation_type.upper()} ({NUM_DRONES} drones)', fontsize=14, fontweight='bold')

    # Gráfico 1: Convergencia del fitness
    ax1.plot(fitness_history, color='royalblue', linewidth=2)
    ax1.set_title('Convergencia del Fitness (gBest)')
    ax1.set_xlabel('Iteración')
    ax1.set_ylabel('Mejor Fitness')
    ax1.grid(True, alpha=0.4)
    ax1.set_yscale('log')
    ax1.annotate(f'Final: {fitness_history[-1]:.2f}',
                 xy=(len(fitness_history)-1, fitness_history[-1]),
                 xytext=(-80, 20), textcoords='offset points',
                 arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

    # Gráfico 2: Formación final de drones
    ax2.set_xlim(0, AREA_WIDTH)
    ax2.set_ylim(0, AREA_HEIGHT)
    ax2.set_aspect('equal')
    ax2.set_title(f'Formación Final — Fitness: {fitness_history[-1]:.2f}')
    ax2.set_xlabel('Coordenada X (m)')
    ax2.set_ylabel('Coordenada Y (m)')
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor('#f8f9fa')

    # Centroide final
    centroid = np.mean(final_positions, axis=0)
    ax2.scatter(*centroid, color='green', marker='x', s=200, zorder=5, label='Centroide', linewidths=3)

    # Posiciones ideales
    ideal_pos = centroid + target_rel_pos
    ax2.scatter(ideal_pos[:, 0], ideal_pos[:, 1], s=200, facecolors='none',
                edgecolors='gray', linewidths=2, zorder=4, label='Posición ideal')

    # Posiciones reales de drones
    colors = plt.cm.tab10(np.linspace(0, 1, NUM_DRONES))
    for i in range(NUM_DRONES):
        ax2.scatter(*final_positions[i], s=150, color=colors[i], zorder=6, label=f'D{i}')
        ax2.text(final_positions[i, 0] + 1.5, final_positions[i, 1] + 1.5, f'D{i}', fontsize=8)
        # Círculo de seguridad
        circle = plt.Circle(final_positions[i], MIN_SAFE_DISTANCE,
                             color=colors[i], fill=False, linestyle='--', alpha=0.5)
        ax2.add_artist(circle)

    ax2.legend(loc='upper right', fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()

print('Función de visualización definida ✓')

In [ ]:
# ============================================================
# EJECUCIÓN: FORMACIÓN EN LÍNEA
# ============================================================
np.random.seed(42)  # Para reproducibilidad
pos_line, hist_line, tgt_line = run_pso('line')
visualize_results(pos_line, hist_line, tgt_line, 'line')

In [ ]:
# ============================================================
# EJECUCIÓN: FORMACIÓN EN H
# ============================================================
np.random.seed(42)
pos_H, hist_H, tgt_H = run_pso('H')
visualize_results(pos_H, hist_H, tgt_H, 'H')

In [ ]:
# ============================================================
# EJECUCIÓN: FORMACIÓN RECTANGULAR
# ============================================================
np.random.seed(42)
pos_rect, hist_rect, tgt_rect = run_pso('rectangle')
visualize_results(pos_rect, hist_rect, tgt_rect, 'rectangle')

In [ ]:
# ============================================================
# ANÁLISIS: IMPACTO DE LOS PARÁMETROS PSO
# Se varía W (inercia) y se compara convergencia
# ============================================================

print('Experimentando con diferentes valores de W (inercia)...')
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Impacto del Parámetro W (Inercia) en la Convergencia PSO — Formación H', fontsize=12)

W_values   = [0.3, 0.5, 0.7, 0.9]
colores    = ['blue', 'green', 'orange', 'red']
target_exp = get_target_formation_relative_positions('H')

for W_test, color in zip(W_values, colores):
    np.random.seed(42)
    # Inicialización
    p_pos = np.random.rand(NUM_PARTICLES, NUM_DRONES * DIMENSIONS)
    p_pos[:, ::2]  *= AREA_WIDTH
    p_pos[:, 1::2] *= AREA_HEIGHT
    p_vel    = np.random.rand(NUM_PARTICLES, NUM_DRONES * DIMENSIONS) * V_MAX * 0.1
    pb_pos   = np.copy(p_pos)
    pb_fit   = np.full(NUM_PARTICLES, float('inf'))
    gb_pos   = np.zeros(NUM_DRONES * DIMENSIONS)
    gb_fit   = float('inf')
    history  = []

    for it in range(MAX_ITERATIONS):
        for i in range(NUM_PARTICLES):
            cf = fitness_function(p_pos[i], target_exp)
            if cf < pb_fit[i]: pb_fit[i] = cf; pb_pos[i] = np.copy(p_pos[i])
            if cf < gb_fit:    gb_fit = cf;     gb_pos    = np.copy(p_pos[i])
        history.append(gb_fit)
        for i in range(NUM_PARTICLES):
            r1, r2 = np.random.rand(NUM_DRONES*DIMENSIONS), np.random.rand(NUM_DRONES*DIMENSIONS)
            p_vel[i] = W_test * p_vel[i] + C1*r1*(pb_pos[i]-p_pos[i]) + C2*r2*(gb_pos-p_pos[i])
            p_vel[i] = np.clip(p_vel[i], -V_MAX, V_MAX)
            p_pos[i] += p_vel[i]
            p_pos[i, ::2]  = np.clip(p_pos[i, ::2],  0, AREA_WIDTH)
            p_pos[i, 1::2] = np.clip(p_pos[i, 1::2], 0, AREA_HEIGHT)

    ax.plot(history, label=f'W={W_test}  (final: {gb_fit:.1f})', color=color, linewidth=2)

ax.set_xlabel('Iteración')
ax.set_ylabel('Mejor Fitness (gBest)')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print('\nConclusión sobre el parámetro W:')
print('  W bajo  (0.3): converge más rápido pero puede quedar en óptimos locales')
print('  W medio (0.5): balance entre exploración y explotación (recomendado)')
print('  W alto  (0.9): explora más el espacio pero converge más lento')

In [ ]:
# ============================================================
# COMPARATIVA FINAL DE LAS 3 FORMACIONES
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparativa de Formaciones — PSO con 6 Drones', fontsize=14, fontweight='bold')

formaciones = [
    ('line',      pos_line, tgt_line,  hist_line,  axes[0]),
    ('H',         pos_H,    tgt_H,     hist_H,     axes[1]),
    ('rectangle', pos_rect, tgt_rect,  hist_rect,  axes[2]),
]

for fname, pos, tgt, hist, ax in formaciones:
    centroid  = np.mean(pos, axis=0)
    ideal_pos = centroid + tgt

    ax.set_xlim(0, AREA_WIDTH)
    ax.set_ylim(0, AREA_HEIGHT)
    ax.set_aspect('equal')
    ax.set_title(f'{fname.upper()}  (Fitness: {hist[-1]:.1f})', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#f0f0f0')

    ax.scatter(ideal_pos[:, 0], ideal_pos[:, 1], s=200, facecolors='none',
               edgecolors='gray', linewidths=2, zorder=4, label='Ideal')

    colors = plt.cm.tab10(np.linspace(0, 1, NUM_DRONES))
    for i in range(NUM_DRONES):
        ax.scatter(*pos[i], s=160, color=colors[i], zorder=6)
        ax.text(pos[i, 0]+1.5, pos[i, 1]+1.5, f'D{i}', fontsize=8)
        circle = plt.Circle(pos[i], MIN_SAFE_DISTANCE, color=colors[i], fill=False, linestyle='--', alpha=0.4)
        ax.add_artist(circle)

    ax.scatter(*centroid, color='red', marker='*', s=200, zorder=7, label='Centroide')

plt.tight_layout()
plt.show()

---
## 📊 Análisis y Conclusiones

### PARTE 1 — Lógica Difusa

**Problema 1 — Control de derrape:**
- La lógica difusa permitió modelar una situación de alta incertidumbre donde ni el ángulo de derrape ni la respuesta correcta son valores exactos.
- **Caso A (Tracción Delantera):** La acción principal es la corrección del volante. El frenado debe ser suave o nulo para evitar bloquear las ruedas directrices.
- **Caso B (Tracción Trasera):** La prioridad es reducir el acelerador. La corrección del volante se hace hacia el lado del derrape (contravolante). El freno es casi siempre contraproducente durante el derrape activo.
- Las funciones de pertenencia trapezoidales fueron adecuadas para modelar rangos fijos (velocidad alta, humedad muy alta), mientras que las triangulares sirvieron para estados de transición.

**Problema 2 — Riesgo crediticio:**
- El sistema difuso demostró que variables como "ingreso alto" o "buen historial" son inherentemente difusas y se modelan mejor con grados de pertenencia que con umbrales binarios.
- La combinación de tres antecedentes permitió generar decisiones matizadas (aprobado / condicional / rechazado) en lugar de respuestas rígidas.

### PARTE 2 — PSO para Drones

**Función de Fitness:**
- Calcula correctamente el error de formación usando el centroide actual como referencia → la formación es independiente de la posición absoluta del grupo.
- La penalización por colisión (10.000 fija) es efectiva: ninguna solución final presentó colisiones.
- Los límites del área de vuelo se aplican con doble mecanismo: penalización en fitness + clipping de posiciones.

**Convergencia:**
- Las tres formaciones convergen satisfactoriamente en menos de 200 iteraciones.
- La formación en H es la más compleja por sus restricciones geométricas simétricas.

**Impacto de parámetros:**
- **W = 0.5** ofrece el mejor balance entre exploración y explotación.
- W elevado (0.9) mejora la exploración inicial pero enlentece la convergencia.
- W bajo (0.3) converge rápido pero puede estancarse en óptimos locales.

**Formación más eficiente:**
- La formación **rectangular** tiende a ser la más eficiente en vuelo real porque:
  1. Minimiza turbulencias entre drones (mayor separación lateral)
  2. Permite cobertura de área rectangular (útil en vigilancia)
  3. Es más estable ante perturbaciones externas (viento)
  4. El PSO la alcanza con fitness similar al de la línea pero con mejor distribución espacial.